## 汇总扩展数据集
### 区分train\test\val数据集 8：1：1
### shuffle

In [ ]:
import glob
import os

import json
import random
import json
from collections import OrderedDict

# 读取JSONL文件并返回数据列表
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# 划分数据集
def split_data(data, split_log, train_ratio=0.8, test_ratio=0.1, dev_ratio=0.1):
    assert train_ratio + test_ratio + dev_ratio == 1 #"比例之和必须为1"
    random.seed(42)
    # 随机打乱数据
    random.shuffle(data)
    
    # 计算每个集合的大小
    total_size = len(data)
    train_size = int(total_size * train_ratio)
    test_size = int(total_size * test_ratio)
    
    # 划分数据集
    train_data = data[:train_size]
    test_data = data[train_size:train_size + test_size]
    dev_data = data[train_size + test_size:]
    sft_data = train_data + dev_data
    assert len(train_data) + len(test_data) + len(dev_data) == total_size
    print("train: ", len(train_data), "test: ", len(test_data), "dev: ", len(dev_data))
    with open(split_log+"split_log.txt",'w',encoding='utf-8') as f_in:
        f_in.write(f"train_len:{len(train_data)}, train_id: 0-{train_size-1}"+ '\n')
        f_in.write(f"test_len:{len(test_data)}, test_id: {train_size}-{train_size + test_size-1}"+ '\n')
        f_in.write(f"dev_len:{len(dev_data)}, dev_id: {train_size + test_size}-{total_size}"+ '\n')
        f_in.write(f"sft_len:{len(sft_data)}, sft_id: 0-{train_size-1}, {train_size + test_size}-{total_size}"+ '\n')
        

    return train_data, test_data, dev_data, sft_data

# 文件路径
# file_path = 'your_data.jsonl'

# 读取数据
# data = read_jsonl(file_path)

# 划分数据集
# train_data, test_data, dev_data = split_data(data)

# 可选：将划分的数据集写入新的JSONL文件
def write_jsonl(data, file_name):
    with open(file_name, 'w',encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item,ensure_ascii=False) + '\n')



def read_all_jsonl_files_from_onedir(generation_dir, out_name, merge_CONTENT):
    all_data_keep = []
    idx = 0
    # all_data_list = []
    if os.path.exists(generation_dir):
        # 查看当前目录下是否都有有一部分文件存在，然后读取已有的生成数据（防止代码中途报错，使得已有的生成数据不要被浪费）
        # 构建搜索模式以匹配所有 .jsonl 文件
        pattern = os.path.join(generation_dir, '*.jsonl')
        # 使用 glob.glob() 获取所有匹配的文件名
        jsonl_files = glob.glob(pattern)
        # 如果存在文件个数大于1，则对已有文件进行已有数据的读取
        # 根据已有文件数据，更新all_task中的数据

        if len(jsonl_files) > 0:
            for filename in jsonl_files:
                with open(filename,'r',encoding='utf-8') as fin:
                    lines = fin.readlines()
                    
                    for line in lines: # 对于每一个章节chapter
                        data  = json.loads(line)
                        # data1 = json.loads(line)
                        
                        if not merge_CONTENT:
                        # 您想保留的键
                            keys_to_keep = ['chapter', 'explanation', 'title','introduction','main_body','conclusion']
                        # 使用字典推导式创建一个新字典，只包含您想保留的键
                            filtered_data = {k: v for k, v in data.items() if k in keys_to_keep}
                            filtered_data['id'] = idx + 1
                            filtered_data = OrderedDict(
                            (k, filtered_data[k]) for k in \
                            ["id", "chapter", "explanation",'title','introduction','main_body','conclusion']
                            )
                            all_data_keep.append(filtered_data)
                        else:
                            keys_to_keep = ['chapter', 'explanation', 'title']
                        # 使用字典推导式创建一个新字典，只包含您想保留的键
                            filtered_data = {k: v for k, v in data.items() if k in keys_to_keep}
                            filtered_data['story_content'] = data['introduction']+"\n"+data['main_body']+"\n"+data['conclusion']
                            filtered_data['id'] = idx + 1
                            filtered_data = OrderedDict(
                            (k, filtered_data[k]) for k in \
                            ["id", "chapter", "explanation",'title','story_content']
                            )
                            all_data_keep.append(filtered_data)
                        idx += 1
                
    write_jsonl(all_data_keep, out_name+".jsonl")
    print("这个目录下的数据数量为：",len(all_data_keep))

    return all_data_keep


gpt4_story_dir_from_gpt4titles =  "SS-GEN/hierarchical_instruct/data/gpt4_test_generations/Stories generation/Generated_Titles_from_gpt4"
out_path = "SS-GEN/hierarchical_instruct/data/SS-GEN Dataset/"
out_name = "original_gpt4story_all_5085_from_gpt4titles"
os.makedirs(out_path, exist_ok=True)# 目录存在也不会报错
all_data_keep = read_all_jsonl_files_from_onedir(gpt4_story_dir_from_gpt4titles , out_path+out_name, True)

train,test,dev,sft = split_data(all_data_keep, out_path+out_name, train_ratio=0.8, test_ratio=0.1, dev_ratio=0.1)
# write_jsonl(all_data_keep, out_name+".jsonl")
write_jsonl(train, out_path+out_name+"_train.jsonl")
write_jsonl(test, out_path+out_name+"_test.jsonl")
write_jsonl(dev, out_path+out_name+"_dev.jsonl")
write_jsonl(sft, out_path+out_name+"_sft.jsonl")